In [ ]:
# Cell 1 -- Setup + download ap-matched-sentences.db
#
# No manual steps needed. Just run this cell.
# Downloads only the .db file from the shared Drive folder,
# deletes .pt files immediately to save disk space.
#
# Requires: Wikipedia training checkpoint (wiki_model.pt)
#           Run kaggle_notebook.ipynb first if missing.

!pip install transformers scikit-learn scipy gdown -q

import os, sys, glob, shutil
from pathlib import Path

# Clone / pull repo
REPO = Path('/kaggle/working/multihop-retrieval')
if REPO.exists():
    os.system(f'git -C {REPO} pull')
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {REPO}')
sys.path.insert(0, str(REPO / 'delta_system'))
print('Repo OK')

# Show checkpoints
ckpts = glob.glob('/kaggle/working/checkpoints/*.pt')
print('Checkpoints:', ckpts)
print()

# ---- Download ap-matched-sentences.db ----
FOLDER_ID = '1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5'   # your shared Drive folder
DB_PATH   = Path('/kaggle/working/newsedits/ap-matched-sentences.db')
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

if DB_PATH.exists() and DB_PATH.stat().st_size > 1e6:
    print(f'Already downloaded: {DB_PATH} ({DB_PATH.stat().st_size/1e6:.0f} MB)')
else:
    import gdown, requests, re

    # Step 1: try to find the individual file ID from the folder page
    print('Looking up file ID for ap-matched-sentences.db...')
    file_id = None
    try:
        resp = requests.get(
            f'https://drive.google.com/drive/folders/{FOLDER_ID}',
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=15
        )
        text = resp.text
        # Google Drive embeds file metadata as JSON in the page source
        # File IDs appear as long base58-like strings starting with 1 or 0
        target = 'ap-matched-sentences'
        idx = text.find(target)
        if idx > 0:
            window = text[max(0, idx-500): idx+500]
            candidates = re.findall(r'"(1[A-Za-z0-9_\-]{25,50})"', window)
            if candidates:
                file_id = candidates[0]
                print(f'Found file ID: {file_id}')
    except Exception as e:
        print(f'Folder page lookup failed: {e}')

    if file_id:
        # Download just this one file
        print('Downloading ap-matched-sentences.db (~406 MB)...')
        gdown.download(id=file_id, output=str(DB_PATH), quiet=False)
    else:
        # Fallback: download whole folder, keep only .db
        print('File ID not found. Downloading full folder (~4.4 GB)...')
        print('This takes ~5-10 minutes on Kaggle. .pt files will be deleted after.')
        TEMP = Path('/kaggle/working/gdrive_temp')
        TEMP.mkdir(exist_ok=True)
        gdown.download_folder(id=FOLDER_ID, output=str(TEMP),
                              quiet=False, remaining_ok=True)

        # Find and move the .db file
        db_files = list(TEMP.rglob('*.db'))
        if db_files:
            shutil.copy(str(db_files[0]), str(DB_PATH))
            print(f'Copied: {db_files[0]} -> {DB_PATH}')
        else:
            print('ERROR: .db file not found in downloaded folder.')
            sys.exit(1)

        # Delete .pt files to free up disk space
        for f in TEMP.rglob('*.pt'):
            f.unlink()
            print(f'Deleted: {f.name}')

print()
print(f'DB ready: {DB_PATH.exists()}  |  Size: {DB_PATH.stat().st_size/1e6:.0f} MB')

In [ ]:
# Cell 2 -- Run zero-shot NewsEdits evaluation
# Wikipedia-trained G evaluated on AP news revision pairs
# No news training at all -- pure domain transfer

DB_PATH = '/kaggle/working/newsedits/ap-matched-sentences.db'

!python /kaggle/working/multihop-retrieval/delta_system/newsedits_zeroshot_eval.py \
    --db {DB_PATH} \
    --ckpt /kaggle/working/checkpoints/wiki_model.pt \
    --n 1000 \
    --min_added 2